In [ ]:
import json
import pandas as pd
from tqdm import tqdm

In [ ]:
# DNA repair genes
genes_df = pd.read_csv('/mnt/project/DNA repair genes/repair_genes_map.txt', 
                       sep = ' ', header = None)

transcript_gene = dict(zip(genes_df[2], genes_df[0]))
transcript_id = dict(zip(genes_df[2], genes_df[1]))

genes_df.head()

In [ ]:
genes_filtered = {}

genes_filtered_df = pd.read_csv('repair_genes_map_filtered.txt', sep = ' ', header = None)

for _, row in genes_filtered_df.iterrows():
    if row[3] == 'X':
        continue
    genes_filtered[row[0]] = True

In [ ]:
# Revel predictions
revel_df = pd.read_csv('/mnt/project/DNA repair genes/revel-v1.3_all_chromosomes.zip', compression = 'zip')

revel_df = revel_df[revel_df['grch38_pos'] != '.']

revel_df = revel_df[revel_df['REVEL'] > 0.75] # deleterious

revel_df['Variant'] = (
    'chr' + revel_df['chr'].astype(str) + ':' +
    revel_df['grch38_pos'].astype(str) + ':' +
    revel_df['ref'].astype(str) + ':' +
    revel_df['alt'].astype(str)
)

mut_revel = dict(zip(revel_df['Variant'], revel_df['REVEL']))

revel_df.head()

In [ ]:
consequence_priority = {
    'transcript_ablation': 0,
    'splice_acceptor_variant': 0,
    'splice_donor_variant': 0,
    'stop_gained': 0,
    'frameshift_variant': 0,
    'missense_variant': 1,
}

def most_severe_consequence(consequence_terms):
    return min(consequence_terms, key = lambda c: consequence_priority.get(c, 2))

In [ ]:
def deleterious_mutations(chrom):
    
    vep_df = pd.read_csv(f'/mnt/project/DNA repair genes/VEP/ukb_chr{chrom}_updated.tsv.bgz', sep = '\t', compression = 'gzip')

    muts = {}

    for _, row in vep_df.iterrows():

        mut = f'chr{row['f0']}:{row['f1']}:{row['f2']}:{row['f3']}'

        vep = json.loads(row['vep'])
        if vep['transcript_consequences']:
            for t in vep['transcript_consequences']:
                if t['gene_id'] in genes_df[1].values:
                    muts[mut] = {}
                    muts[mut]['consequence'] = most_severe_consequence(t['consequence_terms'])
                    muts[mut]['gene_symbol'] = t['gene_symbol']
                    muts[mut]['gene_id'] = t['gene_id']
                    break
                
    lofs = []
    for mut in muts:
        ref = mut.split(':')[2]
        alt = mut.split(':')[3]
        if muts[mut]['consequence'] in ['transcript_ablation', 'splice_acceptor_variant', 'splice_donor_variant', 
                                        'stop_gained', 'frameshift_variant']:
            lofs.append((mut, muts[mut]['consequence'], muts[mut]['gene_symbol'], muts[mut]['gene_id']))
            
    miss = []
    for mut in muts:
        if muts[mut]['consequence'] == 'missense_variant' and mut in mut_revel:
            miss.append((mut, muts[mut]['gene_symbol'], muts[mut]['gene_id']))
            
    return lofs, miss

In [ ]:
chroms = list(range(1, 21)) + [22]

total_lofs = []
total_miss = []

for _, chrom in tqdm(enumerate(chroms)):
    curr_lofs, curr_miss = deleterious_mutations(chrom)
    total_lofs.extend(curr_lofs)
    total_miss.extend(curr_miss)
    
print('Total', len(total_lofs), len(total_miss))

In [ ]:
with open('repair_genes_lofs.txt', 'w') as f:
    for mut, _, _, _ in total_lofs:
        f.write(f'DRAGEN:{mut}\n')
        
with open('repair_genes_lofs_map.txt', 'w') as f:
    for mut, _, gene, _ in total_lofs:
        f.write(f'{mut} {gene}\n')
        
with open('repair_genes_miss.txt', 'w') as f:
    for mut, _, _ in total_miss:
        f.write(f'DRAGEN:{mut}\n')
        
with open('repair_genes_miss_map.txt', 'w') as f:
    for mut, gene, _ in total_miss:
        f.write(f'{mut} {gene}\n')

In [ ]:
!dx upload *.txt --path='DNA repair genes/'